# TrainLM: HF causal LM training on TPU v5e-1 / v5e-8

Run top to bottom from a **fresh Kaggle TPU session**. Set `TRAINLM_EXPECTED_WORLD_SIZE=1` for v5e-1 or leave the default `8` for v5e-8. Jupyter coordinates subprocesses; only workers import Torch/XLA and own TPU devices. The workflow is installation ? collective probe ? model preflight ? binary data ? two optimizer updates ? 100-update measurement ? saved evidence and optional HF export.

The default model is the 135,611,392-parameter HF Llama reference, sequence 2048, batch 2 per replica, accumulation 32. This runs TrainLM's current generic loss and trainer. It does not certify fused kernels, memory-efficient logits, or LaughLM performance parity.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_DIR = Path(os.environ.get("TRAINLM_REPO_DIR", "/kaggle/working/TrainLM")).resolve()
if not (REPO_DIR / "scripts/trainlm_tpu_worker.py").is_file():
    raise FileNotFoundError(f"Place the updated TrainLM checkout at {REPO_DIR}")
os.chdir(REPO_DIR)
print("checkout:", REPO_DIR)

# Set TRAINLM_INSTALL=1 (or set INSTALL=True) once on a fresh kernel.
INSTALL = os.environ.get("TRAINLM_INSTALL", "0") == "1"
if INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[tpu-xla]",
                    "-c", "constraints/tpu-xla-2.9.txt"], check=True)
    raise RuntimeError("Installation finished. Restart the kernel, set INSTALL=False, and rerun.")


## Host environment

Do not run an earlier device-discovery or in-process model cell. A previous XLA client can keep TPU resources alive even after deleting Python variables. Restarting the notebook kernel releases that client. The TensorFlow installation warning alone does not establish the cause of a native allocation failure; retain the complete worker log.


In [ ]:
import importlib.metadata as metadata
import importlib.util
import platform
import shlex
import json
from datetime import datetime, timezone

loaded_accelerators = sorted(set(sys.modules) & {"torch", "torch_xla", "jax", "tensorflow", "transformers"})
if loaded_accelerators:
    raise RuntimeError(f"Restart the kernel before this workflow; already imported: {loaded_accelerators}")
# Kaggle may preload a torchvision/torchaudio build for another Torch release.
# This is text-only training, so remove those optional native extensions before
# Transformers imports its optional vision helpers. Restart once if removed.
REMOVE_OPTIONAL_VISION = True
if REMOVE_OPTIONAL_VISION:
    optional = []
    for _package in ("torchvision", "torchaudio"):
        try:
            _version = metadata.version(_package)
        except metadata.PackageNotFoundError:
            continue
        optional.append((_package, _version))
    if optional:
        print("Removing text-only incompatible packages:", optional)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], check=True)
        raise RuntimeError("Restart the kernel and rerun from the top.")
if REMOVE_OPTIONAL_VISION:
    remaining_optional = [name for name in ("torchvision", "torchaudio") if importlib.util.find_spec(name) is not None]
    if remaining_optional:
        raise RuntimeError(f"Optional native extensions still present after cleanup: {remaining_optional}. Restart and rerun.")
def _distribution_version(name: str, *, required: bool = False) -> str:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        if name == "trainlm":
            # The checkout can exist without editable-install metadata.
            # Install the complete pinned TPU extra, not metadata alone.
            subprocess.run([sys.executable, "-m", "pip", "install",
                            "-e", f"{REPO_DIR}[tpu-xla]", "-c",
                            str(REPO_DIR / "constraints/tpu-xla-2.9.txt")],
                           check=True)
            return metadata.version(name)
        if required:
            raise RuntimeError(f"Required distribution is missing: {name}")
        return "not-installed"

versions = {
    name: _distribution_version(name, required=name in {"torch", "torch-xla", "transformers"})
    for name in ("torch", "torch-xla", "transformers", "libtpu", "trainlm")
}
print({"python": platform.python_version(), **versions})

EXPECTED_WORLD_SIZE = int(os.environ.get("TRAINLM_EXPECTED_WORLD_SIZE", "8"))
if EXPECTED_WORLD_SIZE not in (1, 8):
    raise ValueError("TRAINLM_EXPECTED_WORLD_SIZE must be 1 (v5e-1) or 8 (v5e-8).")
SEQ_LEN = 2048
MICRO_BATCH_PER_DEVICE = 2
GRADIENT_ACCUMULATION_STEPS = 32
SMOKE_STEPS = 2
TRAIN_STEPS = int(os.environ.get("TRAINLM_TRAIN_STEPS", "100"))
WARMUP_STEPS = 5
MODEL_ID = os.environ.get("TRAINLM_MODEL_ID", "")  # Empty = HF Llama reference from config.
MODEL_REVISION = os.environ.get("TRAINLM_HF_REVISION", "")
TRUST_REMOTE_CODE = False
EXPORT_HF = False
CACHE_DIR = Path("/tmp/trainlm_xla_cache")
RUN_DIR = Path("runs") / (f"trainlm_world{EXPECTED_WORLD_SIZE}_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
RUN_DIR.mkdir(parents=True, exist_ok=True)
assert TRAIN_STEPS > WARMUP_STEPS
print(f"Expected TPU world size: {EXPECTED_WORLD_SIZE}")
print("Scheduled tokens/update:", SEQ_LEN * MICRO_BATCH_PER_DEVICE * GRADIENT_ACCUMULATION_STEPS * EXPECTED_WORLD_SIZE)
print("Evidence directory:", RUN_DIR.resolve())


In [ ]:
def launch_worker(arguments, log_name):
    # No TPU imports or device calls in this kernel. The script configures its
    # single-VM environment before importing XLA; spawned workers retain PJRT's ranks.
    command = [sys.executable, "-u", "-X", "faulthandler",
               str(REPO_DIR / "scripts/trainlm_tpu_worker.py"), *arguments]
    child_env = os.environ.copy()
    child_env.update(PYTHONUNBUFFERED="1", PYTHONFAULTHANDLER="1")
    log_path = RUN_DIR / log_name
    print(shlex.join(command))
    with log_path.open("w", encoding="utf-8") as log:
        with subprocess.Popen(command, cwd=REPO_DIR, env=child_env,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
    if returncode:
        raise RuntimeError(f"Worker exited {returncode}. Full stage log: {log_path.resolve()}. "
                           "Stop here and inspect the last stage; no smaller fallback was launched.")
    return log_path

# This must print probe_passed for every expected rank and all_workers_finished.
# It executes a tensor all-reduce without any model/data.
probe_log = launch_worker(["--probe-only", "--expected-world-size", str(EXPECTED_WORLD_SIZE), "--cache-dir", str(CACHE_DIR)], "probe.log")
probe_text = probe_log.read_text(encoding="utf-8")
if probe_text.count('\"stage\": \"probe_passed\"') != EXPECTED_WORLD_SIZE:
    raise RuntimeError(f"Probe did not report probe_passed for all {EXPECTED_WORLD_SIZE} ranks.")


## Packed binary data

Default: download your `laughlm-v1` shards, pin the resolved HF dataset commit, then fully scan the raw payload once per run. No Hub write or sidecar upload is needed. The supplied metadata declares 250,000,000 uint16 tokens per shard and vocab 32011 (Phi-3.5 tokenizer); byte order below is explicitly little-endian. Keep that tokenizer mapping for this from-scratch model. A pretrained model must use the same token-ID mapping, not merely a large enough vocabulary.

For existing TrainLM manifests choose `DATA_MODE="local"`. For other raw formats set the actual dtype/header/count from their producer; do not reuse these values blindly. Document boundaries are unavailable in raw mode, so training uses a continuous causal token stream across documents.


In [ ]:
DATA_MODE = os.environ.get("TRAINLM_DATA_MODE", "raw")
MANIFEST_DIR = Path(os.environ.get("TRAINLM_MANIFEST_DIR", "data/packed/train"))
HF_DATASET_REPO = "LaughTaleAI/LaughLM-Tokenized-Fine"
HF_DATASET_REVISION = os.environ.get("TRAINLM_DATASET_REVISION", "")
HF_DATASET_ROOT = "laughlm-v1"
HF_SHARD_START = 0
HF_SHARD_COUNT = 1
HF_CACHE_DIR = Path("/tmp/laughlm_hf_cache")
# To use mounted local raw shards, list their paths here and skip downloading.
RAW_BIN_PATHS = []
TOKEN_DTYPE = "uint16"
TOKEN_VOCAB_SIZE = 32011
EXPECTED_TOKEN_COUNT = 250_000_000
RAW_HEADER_BYTES = None

if DATA_MODE == "raw":
    if not RAW_BIN_PATHS:
        from huggingface_hub import HfApi, hf_hub_download
        HF_DATASET_REVISION = HfApi().repo_info(
            HF_DATASET_REPO, repo_type="dataset", revision=HF_DATASET_REVISION or "main",
        ).sha
        RAW_BIN_PATHS = [Path(hf_hub_download(
            repo_id=HF_DATASET_REPO, repo_type="dataset", revision=HF_DATASET_REVISION,
            filename=f"{HF_DATASET_ROOT}/laughlm-v1_shard_{index:05d}.bin",
            cache_dir=str(HF_CACHE_DIR),
        )) for index in range(HF_SHARD_START, HF_SHARD_START + HF_SHARD_COUNT)]
    RAW_BIN_PATHS = [Path(path).resolve(strict=True) for path in RAW_BIN_PATHS]
    if RAW_HEADER_BYTES is None:
        # Derive only from the declared exact token count and dtype width,
        # never from sample token values. Unknown layouts require an explicit header.
        width = {"uint16": 2, "uint32": 4, "int32": 4, "int64": 8}[TOKEN_DTYPE]
        headers = {p.stat().st_size - EXPECTED_TOKEN_COUNT * width for p in RAW_BIN_PATHS}
        if len(headers) != 1 or next(iter(headers)) not in (0, 1024):
            raise ValueError(f"File sizes disagree with supplied metadata: header candidates {headers}")
        RAW_HEADER_BYTES = headers.pop()
    DATA_ARGS = ["--data-mode", "raw", "--header-bytes", str(RAW_HEADER_BYTES),
                 "--token-dtype", TOKEN_DTYPE, "--byte-order", "little",
                 "--token-vocab-size", str(TOKEN_VOCAB_SIZE),
                 "--expected-token-count", str(EXPECTED_TOKEN_COUNT)]
    for path in RAW_BIN_PATHS:
        DATA_ARGS += ["--bin-path", str(path)]
    print({"header_bytes": RAW_HEADER_BYTES, "dataset_revision": HF_DATASET_REVISION,
           "shards": [str(p) for p in RAW_BIN_PATHS]})
elif DATA_MODE == "local":
    if not list(MANIFEST_DIR.glob("*.json")):
        raise FileNotFoundError(f"No TrainLM manifests at {MANIFEST_DIR}")
    DATA_ARGS = ["--data-mode", "local", "--manifest-dir", str(MANIFEST_DIR)]
else:
    raise ValueError("Use raw downloaded/mounted files or local manifests in this notebook.")

COMMON_ARGS = [*DATA_ARGS, "--expected-world-size", str(EXPECTED_WORLD_SIZE),
               "--sequence-length", str(SEQ_LEN),
               "--micro-batch-per-device", str(MICRO_BATCH_PER_DEVICE),
               "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION_STEPS),
               "--cache-dir", str(CACHE_DIR), "--model-id", MODEL_ID,
               "--model-revision", MODEL_REVISION, "--warmup-steps", str(WARMUP_STEPS)]
if TRUST_REMOTE_CODE:
    COMMON_ARGS += ["--trust-remote-code"]
(RUN_DIR / "request.json").write_text(json.dumps({
    "arguments": COMMON_ARGS, "dataset_repo": HF_DATASET_REPO,
    "dataset_revision": HF_DATASET_REVISION, "model_revision": MODEL_REVISION,
}, indent=2) + "\n", encoding="utf-8")


## Model preflight and smoke: all eight replicas

The model preflight first loads the HF model and executes one XLA forward on every rank, stopping before data and optimizer setup. The smoke then performs two complete updates. Labels remain unshifted in the data; the TrainLM task applies the causal shift once. Host preparation precedes asynchronous transfer. This validation uses the generic unrolled GA32 baseline, so the XLA graph boundary is per optimizer update; microstep/xla-loop accumulation remains a later performance milestone.


In [ ]:
preflight_log = launch_worker([*COMMON_ARGS, "--model-preflight", "--output-dir", str(RUN_DIR / "model_preflight")], "model_preflight.log")
preflight_text = preflight_log.read_text(encoding="utf-8")
preflight_passed = (preflight_text.count('\"stage\": \"model_preflight_passed\"') +
                    preflight_text.count("'stage': 'model_preflight_passed'"))
if preflight_passed != EXPECTED_WORLD_SIZE:
    raise RuntimeError(f"Model preflight did not pass on all {EXPECTED_WORLD_SIZE} ranks.")
launch_worker([*COMMON_ARGS, "--max-steps", str(SMOKE_STEPS),
               "--log-every-steps", "1", "--output-dir", str(RUN_DIR / "smoke")], "smoke.log")
smoke = json.loads((RUN_DIR / "smoke/summary.json").read_text())
assert smoke["world_size"] == EXPECTED_WORLD_SIZE and smoke["steps"] == SMOKE_STEPS
assert smoke["phase"] == "finalized"
print(smoke)


## Measured run and optional export

The smoke workers exit before this fresh run. Both runs use identical model/data/precision/geometry. The persistent cache may be warm; the first five updates are excluded from the measured window. Timing waits for device completion and uses the slowest replica's window. The report distinguishes scheduled input tokens from supervised next-token targets (2047 per 2048-token sequence).

This baseline still uses full logits and per-update trainer synchronization. The existing `logits_chunk_size` configuration did not activate chunked loss; this notebook no longer implies it does. Attention/loss kernel work and measured performance certification remain roadmap gates.


In [ ]:
measured_args = [*COMMON_ARGS, "--max-steps", str(TRAIN_STEPS),
                 "--log-every-steps", "10", "--output-dir", str(RUN_DIR / "baseline")]
if EXPORT_HF:
    measured_args += ["--export-hf"]
launch_worker(measured_args, "baseline.log")
summary = json.loads((RUN_DIR / "baseline/summary.json").read_text())
assert summary["world_size"] == EXPECTED_WORLD_SIZE and summary["steps"] == TRAIN_STEPS
assert summary["phase"] == "finalized"
print(json.dumps(summary, indent=2))
print("Keep request.json, probe.log, smoke.log, baseline.log, baseline/summary.json, and baseline/xla_metrics.txt.")
print("HF export is optional weights/config export; it is not an exact-resume optimizer/data checkpoint.")
